In [9]:
import optax
import jax
from jax.nn import sigmoid
from jax.tree_util import tree_map
from typing import Callable
jax.config.update('jax_platform_name', 'cpu')

def scale_parameters(parameters, bounds):
    return {key:sigmoid(parameters[key]) * bounds[key][1] for key in parameters}

def fit(
    loss_fn: Callable, 
    bounds: dict,
    params: optax.Params, 
    optimizer: optax.GradientTransformation
) -> optax.Params:

    # @jax.jit
    def step(state, _):
        params, opt_state = state
        params = scale_parameters(params, bounds)
        loss_value, grads = jax.value_and_grad(loss_fn)(params)
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        return params, opt_state, loss_value

    opt_state = optimizer.init(params)
    state = params, opt_state
    params, opt_state, loss_value = jax.lax.scan(step, state, jax.numpy.arange(1000))
    return params

In [10]:
from jaxcmr.memorysearch import BaseCMR, uniform_presentations_data_likelihood
from jaxcmr.datasets import load_data, generate_trial_mask, load_parameters

model_name = 'Base_CMR'
parameter_path = '../../data/base_cmr_parameters.json'

data_tag = 'HealyKahana2014'
trial_query = "data['listtype'] == -1"

result_path = '../../data/results/jax_lowprec_{}_{}_{}.jsonl'
data_path = '../../data/{}.h5'

ignore_first_recall = False

data = load_data(data_path.format(data_tag))
parameters = load_parameters(parameter_path)
bounds = parameters['free']

list_length = 16
trial_mask = generate_trial_mask(data, trial_query)
trials = data['recalls'][trial_mask]

def loss(parameters):
    model = BaseCMR.create(list_length, parameters)
    return uniform_presentations_data_likelihood(model, trials)

bounds

{'encoding_drift_rate': [2.220446049250313e-16, 0.9999999999999998],
 'delay_drift_rate': [2.220446049250313e-16, 0.9999999999999998],
 'start_drift_rate': [2.220446049250313e-16, 0.9999999999999998],
 'recall_drift_rate': [2.220446049250313e-16, 0.9999999999999998],
 'shared_support': [2.220446049250313e-16, 100.0],
 'item_support': [2.220446049250313e-16, 100.0],
 'learning_rate': [2.220446049250313e-16, 0.9999999999999998],
 'primacy_scale': [2.220446049250313e-16, 100.0],
 'primacy_decay': [2.220446049250313e-16, 100.0],
 'stop_probability_scale': [2.220446049250313e-16, 0.9999999999999998],
 'stop_probability_growth': [2.220446049250313e-16, 10.0],
 'choice_sensitivity': [2.220446049250313e-16, 100.0]}

In [12]:
# Finally, we can fit our parametrized function using the Adam optimizer
# provided by optax.

initial_params = {
    "choice_sensitivity": jax.random.uniform(key=jax.random.PRNGKey(12)),
    "delay_drift_rate": jax.random.uniform(key=jax.random.PRNGKey(2)),
    "encoding_drift_rate": jax.random.uniform(key=jax.random.PRNGKey(1)),
    "item_support": jax.random.uniform(key=jax.random.PRNGKey(6)),
    "learning_rate": jax.random.uniform(key=jax.random.PRNGKey(7)),
    "primacy_scale": jax.random.uniform(key=jax.random.PRNGKey(8)),
    "primacy_decay":jax.random.uniform(key=jax.random.PRNGKey(9)),
    "recall_drift_rate": jax.random.uniform(key=jax.random.PRNGKey(4)),
    "shared_support": jax.random.uniform(key=jax.random.PRNGKey(5)),
    "start_drift_rate": jax.random.uniform(key=jax.random.PRNGKey(3)),
    "stop_probability_growth": jax.random.uniform(key=jax.random.PRNGKey(11)),
    "stop_probability_scale": jax.random.uniform(key=jax.random.PRNGKey(10)),
}

optimizer = optax.adam(learning_rate=1e-2)
params = fit(loss, bounds, initial_params, optimizer)
params

{'choice_sensitivity': Array(nan, dtype=float32),
 'delay_drift_rate': Array(nan, dtype=float32),
 'encoding_drift_rate': Array(nan, dtype=float32),
 'item_support': Array(nan, dtype=float32),
 'learning_rate': Array(nan, dtype=float32),
 'primacy_decay': Array(nan, dtype=float32),
 'primacy_scale': Array(nan, dtype=float32),
 'recall_drift_rate': Array(nan, dtype=float32),
 'shared_support': Array(nan, dtype=float32),
 'start_drift_rate': Array(nan, dtype=float32),
 'stop_probability_growth': Array(nan, dtype=float32),
 'stop_probability_scale': Array(nan, dtype=float32)}

In [17]:
params = scale_parameters(initial_params, bounds)
loss(params)

Array(inf, dtype=float32)

In [16]:
params

{'choice_sensitivity': Array(57.536392, dtype=float32),
 'delay_drift_rate': Array(0.60444516, dtype=float32),
 'encoding_drift_rate': Array(0.5295032, dtype=float32),
 'item_support': Array(62.621506, dtype=float32),
 'learning_rate': Array(0.6993295, dtype=float32),
 'primacy_scale': Array(71.33907, dtype=float32),
 'primacy_decay': Array(65.86118, dtype=float32),
 'recall_drift_rate': Array(0.54044896, dtype=float32),
 'shared_support': Array(65.347305, dtype=float32),
 'start_drift_rate': Array(0.70378613, dtype=float32),
 'stop_probability_growth': Array(6.112258, dtype=float32),
 'stop_probability_scale': Array(0.52233183, dtype=float32)}